<a href="https://colab.research.google.com/github/Minenhlekhuzwayo/Satellite-Data-Assimilation-For-Hybrid-Crop-Yield-Prediction/blob/main/HonoursProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DATA PREPROCESSING

In [ ]:
!pip install refet
import pandas as pd
import refet

# Load data
df_weather = pd.read_csv('weatherDataNASA.csv')

# Remove hidden spaces
df_weather.columns = df_weather.columns.str.strip()

# Rename columns
df_weather = df_weather.rename(columns={
    'DOY':'dateOfYear',
    'ALLSKY_SFC_SW_DWN': 'solarRadiation',
    'T2M_MAX' : 'MaxTemp',
    'T2M_MIN' : 'MinTemp',
    'PRECTOTCORR': 'Precipitation',
    'RH2M': 'RelativeHumidity',
    'WS2M': 'WindSpeed',
    'GWETROOT': 'RootMoisture'
})

# Create Date column
df_weather['Date'] = (
    pd.to_datetime(df_weather['YEAR'], format='%Y') +
    pd.to_timedelta(df_weather['dateOfYear'] - 1, unit='D')
)

#df_weather['Date'] = df_weather['Date'].dt.strftime('%Y/%m/%d')
df_weather['Day'] = df_weather['Date'].dt.day
df_weather['Month'] = df_weather['Date'].dt.month
df_weather['Year'] = df_weather['Date'].dt.year


# Constants
altitude = 350
latitude = -33.5

eto_list = []

# ETo calculation
for index, row in df_weather.iterrows():

    eto = refet.Daily(
        tmin=row['MinTemp'],
        tmax=row['MaxTemp'],
        rs=row['solarRadiation'],
        uz=row['WindSpeed'],
        zw = 2,
        elev=altitude,
        lat=latitude,
        doy=row['dateOfYear'],
        tdew = row['MinTemp']
    ).eto()

    eto_list.append(float(eto)) # Extract the scalar value

# Add ETo column, renaming it to 'ReferenceET' as expected by AquaCrop's prepare_weather
df_weather['ReferenceET'] = eto_list

# 1. Full dataset (for documentation/analysis) with ETo
df_final_full_weather = df_weather[
    ['Date', 'MinTemp', 'MaxTemp', 'Precipitation', 'solarRadiation', 'RelativeHumidity', 'WindSpeed', 'RootMoisture','ReferenceET']
]

# Export full dataset with ETo
df_weather.to_csv('full_weather_with_eto.txt', sep=' ', index=False)



# 2. Final AquaCrop dataframe
df_final_weather_aqua_inputs = df_weather[
    ['Day', 'Month', 'Year','MinTemp', 'MaxTemp', 'Precipitation', 'ReferenceET'] # Use 'ReferenceET' here
]

# Export
df_final_weather_aqua_inputs.to_csv(
    'aquaCropWeatherData.txt',
    sep=' ',
    index=False
)

# Preview
print(df_final_weather_aqua_inputs.head())

RUNNING AQUACROP MODEL - FOR MAIZE

In [ ]:
import os
print(os.listdir())

In [ ]:
!pip install aquacrop
from aquacrop import AquaCropModel, Soil, Crop, InitialWaterContent
from aquacrop.utils import prepare_weather

weather_df = prepare_weather('aquaCropWeatherData.txt')

grape_crop = Crop('Maize', planting_date='10/01')
grape_crop.Maturity=240
grape_crop.Zmax=2.0
grape_crop.CGC=0.004
grape_crop.CDG=0.002
grape_crop.HI0=0.35

model_os = AquaCropModel(
    sim_start_time='2010/01/01',
    sim_end_time='2024/12/31',
    weather_df=weather_df,
    soil=Soil('SandyLoam'),
    crop=grape_crop,
    initial_water_content=InitialWaterContent(value=['FC'])
)

model_os.run_model(till_termination=True)

results = model_os.get_simulation_results() # Returns seasonal summary output

print(results)

# Crop growth outputs
print("--------------------Crop Growth Output--------------------")
crop_growth = model_os._outputs.crop_growth
print(crop_growth)

# Water flux outputs
print("--------------------Water Flux Output---------------------")
water_flux = model_os._outputs.water_flux
print(water_flux)

# Soil water outputs
print("--------------------Water Storage Outputs--------------------")
water_storage = model_os._outputs.water_storage
print(water_storage)

# EXPORTING OUTPUTS
results.to_csv('results_Maize.csv')
crop_growth.to_csv('crop_growth_Maize.csv')
water_flux.to_csv('water_flux_Maize.csv')
water_storage.to_csv('water_storage_Maize.csv')

In [ ]:
print(dir(model_os._outputs))

In [ ]:
crop_growth = model_os._outputs.crop_growth

print(crop_growth.head)

### SENTINEL-2 FEATURE EXTRACTION FOR HYBRIB YIELD PREDICTION

In [ ]:
# FEATURES
# - NDVI
# - EVI
# - NDWI
# - NDRE
# - Raw bands (B4, B5, B8, B11)
# PERIOD:
         # September -> March
# OUTPUT:
         # CSV-ready feature table for ML / data assimilation



# 1. INSTALL & IMPORT LIBRARIES
!pip install earthengine-api geemap
import ee
import pandas as pd


# 2. AUTHENTICATE & INITIALIZE EARTH ENGINE
ee.Authenticate()
ee.Initialize(project='vineyard-yield-project-400201')
print("Earth Engine initialized successfully!")


# 3. STUDY AREA
study_area = ee.Geometry.Rectangle([
    19.30,   # min longitude
    -33.60,  # min latitude
    19.90,   # max longitude
    -33.30   # max latitude
])



# 4. LOADING SENTINEL-2 LEVEL-2A DATA
s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(study_area)
    .filterDate('2015-09-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)



# 5. CLOUD MASKING FUNCTION
def mask_clouds(image):

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    masked =  image.updateMask(mask).divide(10000)
    return masked.copyProperties(
        image,
        image.propertyNames()
    )


# Apply cloud mask FIRST
s2 = s2.map(mask_clouds)




# 6. VEGETATION INDICES CALCULATIONS
def add_indices(image):

    ndvi = image.normalizedDifference(['B8', 'B4']) \
        .rename('NDVI')

    ndwi = image.normalizedDifference(['B8', 'B11']) \
        .rename('NDWI')

    ndre = image.normalizedDifference(['B8', 'B5']) \
        .rename('NDRE')

    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }
    ).rename('EVI')

    # IMPORTANT:
    # add indices back to image
    return image.addBands([ndvi, ndwi, ndre, evi]) \
      .copyProperties(
          image,
          image.propertyNames()
      )


# Apply index calculation
s2_with_indices = s2.map(add_indices)




# 7. SELECTING REQUIRED FEATURES
bands = [
    'NDVI',
    'NDWI',
    'NDRE',
    'EVI',
    'B4',
    'B5',
    'B8',
    'B11'
]




# 8. EXTRACTING MEAN VALUES OVER STUDY AREA
def extract_features(image):

    stats = image.select(bands).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=study_area,
        scale=10,
        maxPixels=1e13
    )

    feature = ee.Feature(
        None,
        stats
    ).set(
        'date',
        image.date().format('YYYY-MM-dd')
    )

    return feature


# Convert images to features
features = s2_with_indices.map(extract_features)

feature_collection = ee.FeatureCollection(features)




# 9. EXPORT TO CSV
task = ee.batch.Export.table.toDrive(
    collection=feature_collection,
    description='Sentinel2_Vineyard_Features',
    folder='satelliteDataFolder',
    fileFormat='CSV'
)

task.start()

print(task.status())

In [ ]:
print(task.status())